# Aprendizagem Automática (APRAU) - Trabalho de Grupo

**Mestrado em Engenharia Informática · ISEP · 2026/2027**

**Grupo 14**

| Nome | Nº de aluno |
|------|-------------|
| [Nome 1] | [Número 1] |
| [Nome 2] | [Número 2] |

**Aluna 1:** Ellah Fleischmann 1231478 · **Aluna 2:** Inês Martinho 1230806
#
**Docente (PL):** Elsa Ferreira Gomes · **Turma:** M1A

---

### Índice
1. Introdução
2. Preparação do ambiente e carregamento dos dados
3. Análise Exploratória de Dados (EDA)
4. Regressão
5. Classificação
6. Seleção de Variáveis (Ridge e Lasso)
7. Conclusões
8. Referências

## 1. Introdução

Este trabalho aplica diferentes métodos de aprendizagem automática a um conjunto de dados clínico multi-doença, com o objetivo de analisar o comportamento dos modelos e interpretar criticamente os resultados.

O dataset provém de um *data warehouse* (DW) médico consolidado, desenhado para estudar padrões de comorbilidade em doenças cardiometabólicas crónicas. Cada registo representa o perfil de um doente, com variáveis demográficas, antropométricas, de estilo de vida e biomarcadores laboratoriais, e tem duas variáveis alvo:

- `target_class` (categórica): categoria de diagnóstico clínico, usada na **classificação**;
- `target_regression` (contínua): medição laboratorial, usada na **regressão**.

O subconjunto atribuído ao grupo 14 tem 2000 registos e três classes de diagnóstico (DI, HT e HT_HY), com forte desbalanceamento entre elas.

**Estrutura do trabalho**

| Secção | Conteúdo |
|--------|----------|
| EDA | Estatística descritiva, análise univariada e bivariada; deteção de variáveis constantes, redundantes ou pouco informativas |
| Regressão | Regressão linear simples e múltipla, comparadas com R², MAE e RMSE |
| Classificação | Regressão logística e LDA, com holdout, CV (k=5 e k=10), LOOCV e bootstrap |
| Seleção de variáveis | Ridge e Lasso: todas as variáveis vs. conjunto reduzido |

## 2. Preparação do ambiente e carregamento dos dados

Nesta secção importamos as bibliotecas, fixamos a semente aleatória (para os resultados serem reprodutíveis) e carregamos o dataset. Fazemos ainda uma verificação inicial da dimensão, dos tipos de dados e da distribuição da variável alvo de classificação.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42 # semente usada em todo o trabalho

In [2]:
df = pd.read_csv("../data/dataset_grupo_14.csv")

print(df.shape)
df.head()

(2000, 35)


,age_level,gender,bmi_level,smoking,age,age_normalized,bmi,HbA1c_level,cholesterol,sleep_hours,...,hdl,ldl,education_level,employment_status,systolic_index,calcium_baseline,plasma_value,ldl_alternative,target_regression,target_class
0,Senior,Male,Overweight,Never,82,0.919540,29.20,7.003349,207.00000,8.000000,...,79.000000,160.000000,Primary,Retired,137.000000,9.42,1.094354,128.408780,194.0,DI
1,Senior,Female,Overweight,Never,80,0.896552,27.32,6.500000,224.41791,7.179453,...,64.012168,129.853982,Primary,Retired,136.462389,9.42,1.339049,97.000000,160.0,DI
2,Senior,Male,Normal,Never,73,0.816091,21.10,7.010714,175.00000,5.900000,...,45.000000,106.000000,Secondary,Employed,133.000000,9.42,1.415402,112.000000,104.0,DI
3,Senior,Female,Obese,Former,85,0.954022,34.80,6.865766,163.00000,5.500000,...,98.000000,116.000000,Tertiary,Employed,95.000000,9.42,1.173173,128.585774,195.0,DI
4,Senior,Male,Overweight,Former,69,0.770114,28.90,6.993220,172.00000,6.900000,...,31.000000,97.000000,Tertiary,Retired,128.000000,9.42,1.210187,152.000000,171.0,DI


### 2.1 Estrutura e tipos de dados

Verificamos os tipos de cada coluna e a existência de valores em falta.

In [3]:
df.info()
print("\nValores em falta por coluna:")
print(df.isnull().sum()[df.isnull().sum() > 0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 35 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age_level             2000 non-null   object 
 1   gender                2000 non-null   object 
 2   bmi_level             2000 non-null   object 
 3   smoking               2000 non-null   object 
 4   age                   2000 non-null   int64  
 5   age_normalized        2000 non-null   float64
 6   bmi                   2000 non-null   float64
 7   HbA1c_level           2000 non-null   float64
 8   cholesterol           2000 non-null   float64
 9   sleep_hours           2000 non-null   float64
 10  triglycerides         2000 non-null   float64
 11  physical_activity     2000 non-null   object 
 12  family_history        2000 non-null   object 
 13  stress_level          2000 non-null   object 
 14  low_hdl_cholesterol   2000 non-null   object 
 15  high_ldl_cholesterol 

### 2.2 Distribuição da variável alvo de classificação

Como o enunciado indica que a maioria dos grupos trabalha com um dataset desbalanceado, é importante perceber desde já a proporção de cada classe, pois isso condiciona a escolha das métricas de avaliação.

In [4]:
print(df['target_class'].value_counts())
print()
print(df['target_class'].value_counts(normalize=True).round(3))

target_class
DI       1714
HT        183
HT_HY     103
Name: count, dtype: int64

target_class
DI       0.857
HT       0.092
HT_HY    0.052
Name: proportion, dtype: float64


### 2.3 Separação dos tipos de variáveis

Separamos as variáveis categóricas das numéricas, excluindo os dois alvos, para as reutilizarmos ao longo da análise.

In [6]:
cat_cols = df.select_dtypes(include=['object', 'string']).columns.drop('target_class').tolist()
num_cols = df.select_dtypes(include=['number']).columns.drop('target_regression').tolist()

print(f"Categóricas: ({len(cat_cols)}) {cat_cols}")
print(f"Numéricas: ({len(num_cols)}) {num_cols}")

Categóricas: (12) ['age_level', 'gender', 'bmi_level', 'smoking', 'physical_activity', 'family_history', 'stress_level', 'low_hdl_cholesterol', 'high_ldl_cholesterol', 'sugar_consumption', 'education_level', 'employment_status']
Numéricas: (21) ['age', 'age_normalized', 'bmi', 'HbA1c_level', 'cholesterol', 'sleep_hours', 'triglycerides', 'blood_pressure', 'crp_level', 'homocysteine_level', 'systolic_bp', 'diastolic_bp', 'alcohol_intake', 'salt_intake', 'heart_rate', 'hdl', 'ldl', 'systolic_index', 'calcium_baseline', 'plasma_value', 'ldl_alternative']
